# ALLaM-7B-Instruct — Query2Doc Query Generator

**Experiment:** exp_007 — ALLaM-7B-Instruct-preview + Dense Retrieval  
**Technique:** Query2Doc (pseudo-document generation)  
**Reference baseline:** exp_003 (Qwen 2.5 3B, NDCG@10=0.5435)

## Model Details
- **Model:** `ALLaM-AI/ALLaM-7B-Instruct-preview` (Arabic-specialized, ~7B params)
- **Architecture:** Standard LlamaForCausalLM (RoPE, SiLU, RMSNorm, Full MHA)
- **Developers:** NCAI at SDAIA (Saudi Data and AI Authority)
- **Training:** 5.2 trillion tokens (4T English + 1.2T Arabic/English), SFT + DPO
- **Vocab:** 64,000 tokens (expanded from Llama-2's 32K for Arabic)
- **Context:** 4,096 tokens
- **Paper:** ICLR 2025, arXiv:2407.15390

## GPU Strategy: A100 (40 GB) — Colab Pro
- **Option A (preferred):** FP16 (~16-17 GB) + batch_size=8 -> preserves full quality
- **Option B (fallback):** 4-bit NF4 (~6-7 GB) + batch_size=8-16 -> if FP16 OOMs
- **Estimated time:** ~30-45 min for 2,896 queries (with batching)

## ALLaM-Specific Notes
- Standard LlamaForCausalLM: NO batching bugs (unlike Falcon-H1's Mamba architecture)
- `apply_chat_template()` works — Llama-2 [INST] style chat template
- MUST use `return_token_type_ids=False` or pop token_type_ids before generate()
- MUST set pad_token manually (not set by default)
- **PREVIEW MODEL:** Pin to `revision="v2"` for reproducibility
- NOT gated — no HuggingFace login required
- Recommended temperature: 0.6 (from model card)

## Key Research Sources
- Paper: Bari et al. (2025). ALLaM: Large Language Models for Arabic and English. ICLR 2025.
- arXiv: https://arxiv.org/abs/2407.15390
- Full research: `research_decisions/allam_7b_research.md`

---

## Step 1: Install Dependencies

> After this cell: Runtime -> Restart runtime, then continue from Step 2.

In [ ]:
# == Step 1: Install all dependencies ========================================
#
# Runtime: A100 (40 GB) -- select in Runtime -> Change runtime type
#
# Java is required by pyserini (for MIRACL data loading).
# bitsandbytes required for 4-bit quantization (fallback if FP16 OOMs).
#
# After this cell: Runtime -> Restart runtime, then continue from Step 2.
# ============================================================================

# 1. Java (required by pyserini -- hidden dependency in MIRACLDataLoader)
!apt-get install -qq openjdk-21-jdk-headless

# 2. Retrieval / data loading libraries
!pip install -q pyserini faiss-cpu

# 3. Transformers (ALLaM uses standard LlamaForCausalLM, requires >= 4.40.1)
!pip install -q transformers

# 4. 4-bit quantization dependencies (for fallback if FP16 OOMs)
!pip install -q bitsandbytes accelerate

# 5. ML / utility libraries
!pip install -q torch datasets tqdm

print("\n" + "=" * 60)
print("Installation complete")
print("=" * 60)
print("IMPORTANT: Restart runtime now!")
print("   Runtime -> Restart runtime")
print("   Then run cells starting from Step 2")
print("=" * 60)

## Step 2: Mount Drive and Setup Environment

> Run this after restarting runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone project repo (or pull if already cloned)
!git clone https://github.com/Osmanoor/graduation.git 2>/dev/null || (cd /content/graduation && git pull)
%cd /content/graduation/arabic-rag-query-enhancement

import os
import sys

# Java home required by pyserini
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

import torch
print(f"\nEnvironment configured")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"Transformers version: {transformers.__version__}")

## Step 3: Load MIRACL Arabic Data

In [ ]:
from src.utils.data_loader import MIRACLDataLoader

data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nDataset Statistics:")
print(f"  Queries: {len(query_ids)}")
print(f"  Qrels:   {len(qrels)}")
print(f"\nSample query: {query_texts[0]}")

## Step 4: Initialize ALLaM-7B-Instruct

**Strategy:** Try FP16 first (preserves full quality). Fall back to 4-bit NF4 if OOM.

**A100 40GB VRAM budget:**
- FP16: ~16-17 GB model -> ~23 GB free for batching -> batch_size=8
- 4-bit: ~6-7 GB model -> ~33 GB free for batching -> batch_size=8-16

**Architecture:** Standard LlamaForCausalLM (RoPE, SiLU, Full MHA with 32 heads)  
No SSM buffers, no batching bugs. Safe to use batch generation.

**PREVIEW MODEL:** Using `revision="v2"` (latest alpha) for reproducibility.  
NOT gated -- no login required.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "ALLaM-AI/ALLaM-7B-Instruct-preview"
MODEL_REVISION = "v2"  # Pin preview model to specific revision for reproducibility

# == Configuration =============================================================
# Strategy: FP16 first (best quality), 4-bit fallback if OOM
USE_4BIT = False  # Set True to force 4-bit, or leave False to try FP16 first
BATCH_SIZE = 8    # Start with 8; adjust based on VRAM report below
# ==============================================================================

print(f"Loading {MODEL_NAME} (revision={MODEL_REVISION})...")
print(f"Mode: {'4-bit NF4' if USE_4BIT else 'FP16 (full quality)'}")
print(f"Target batch size: {BATCH_SIZE}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Required for decoder-only batch generation

# Load model
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        torch_dtype=torch.float16,
        device_map="auto"
    )

model.eval()

# Report VRAM usage
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {used:.1f} GB used / {total/1e9:.1f} GB total")
    print(f"Free: {free/1e9:.1f} GB")
    # Suggest batch size based on free VRAM
    if free / 1e9 > 25:
        suggested_bs = 16
    elif free / 1e9 > 15:
        suggested_bs = 8
    elif free / 1e9 > 8:
        suggested_bs = 4
    else:
        suggested_bs = 1
    print(f"Suggested batch size: {suggested_bs} (based on {free/1e9:.1f} GB free)")
    if suggested_bs != BATCH_SIZE:
        print(f"  -> Consider changing BATCH_SIZE to {suggested_bs}")

print(f"\nALLaM-7B-Instruct ready ({'4-bit NF4' if USE_4BIT else 'FP16'})")
print(f"  Revision: {MODEL_REVISION}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token}")
print(f"  Context length: 4,096 tokens")

import transformers
print(f"  Transformers: {transformers.__version__}")

## Step 5: Sanity Check -- First 5 Queries

**Check before proceeding to full run:**
- Output is in Arabic (not English, not garbage/repetition)
- Pseudo-document is relevant to the query topic
- Expansion ratio is reasonable (5-12x)
- No error messages or warnings

**ALLaM-specific:**
- Remove `token_type_ids` from inputs before `generate()` (same as Jais-2)
- Uses Llama-2 [INST] chat template via `apply_chat_template()`
- Model card recommends temperature=0.6 (lower than our standard 0.7)

In [ ]:
SYSTEM_PROMPT = (
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "Respond in Arabic only."
)

# ALLaM model card recommends temperature=0.6
# We test both 0.6 (model optimal) and 0.7 (cross-model comparison)
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 128
TOP_P = 0.9

print(f"Temperature: {TEMPERATURE}")
print(f"Note: ALLaM recommends temp=0.6. Change TEMPERATURE above to test.")
print("\nSanity check: testing on first 5 queries (single-query mode)\n")
print("=" * 60)

for i in range(5):
    query = query_texts[i]

    # Format using chat template (Llama-2 [INST] style -- works for ALLaM)
    chat_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and remove token_type_ids (ALLaM/Llama requirement)
    inputs = tokenizer(chat_text, return_tensors="pt", return_token_type_ids=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode only the generated tokens (skip the prompt)
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Construct enhanced query (Query2Doc format: original + pseudo-doc)
    enhanced = f"{query} {generated}"
    ratio = len(enhanced) / max(len(query), 1)

    print(f"\nQuery {i+1} [{query_ids[i]}]: {query}")
    print(f"Generated ({len(generated)} chars):")
    print(f"{generated[:300]}..." if len(generated) > 300 else generated)
    print(f"Expansion ratio: {ratio:.1f}x")

print("\n" + "=" * 60)
print("Sanity check complete")
print("\nBefore proceeding, verify:")
print("  [ ] Output is in Arabic")
print("  [ ] Content is relevant to query")
print("  [ ] Expansion ratio 5-12x")
print("  [ ] No errors or garbage output")
print("\nNOTE: If output quality is poor, try TEMPERATURE = 0.6 (ALLaM recommended)")

## Step 6: Full Generation -- 2,896 Queries (Batched)

**Mode:** Batched generation (BATCH_SIZE from Step 4)  
**Why batching works:** ALLaM is standard LlamaForCausalLM -- no Mamba SSM bugs  
**Expected time:** ~30-45 min on A100 with batch_size=8  
**Checkpoints:** Saves progress every 200 queries to pkl  

**Lessons applied:**
- Falcon-H1 (exp_005): Hybrid Mamba architecture forced batch_size=1. ALLaM is standard Transformer.
- Jais-2 (exp_006): Must remove token_type_ids. Same applies to ALLaM.
- Both: OOM fallback handler processes failed batches one query at a time.

In [ ]:
import time
import pickle
from tqdm.notebook import tqdm

CHECKPOINT_EVERY = 200
CHECKPOINT_PATH = 'enhanced_queries_allam_7b_checkpoint.pkl'

def generate_batch(batch_queries):
    """Generate pseudo-documents for a batch of queries.

    Uses left-padded tokenization for parallel generation.
    Uses return_token_type_ids=False (ALLaM/Llama requirement).
    """
    # Format all queries using chat template
    batch_texts = []
    for query in batch_queries:
        chat_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query}
            ],
            tokenize=False,
            add_generation_prompt=True
        )
        batch_texts.append(chat_text)

    # Tokenize with left-padding for batch generation
    # return_token_type_ids=False: ALLaM requirement (same as Jais-2)
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
        return_token_type_ids=False
    ).to(model.device)

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode only the generated tokens (skip prompt)
    generated_texts = tokenizer.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )

    # Combine: original query + pseudo-document
    enhanced = [
        f"{q} {g.strip()}" for q, g in zip(batch_queries, generated_texts)
    ]
    return enhanced


print("=" * 60)
quant_mode = '4-bit NF4' if USE_4BIT else 'FP16'
print(f"FULL RUN: ALLaM-7B-Instruct ({quant_mode}, temp={TEMPERATURE})")
print(f"Revision: {MODEL_REVISION}")
print(f"Queries: {len(query_texts)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Mode: Batched (batch_size={BATCH_SIZE})")
print(f"Checkpoints: every {CHECKPOINT_EVERY} queries")
num_batches = (len(query_texts) + BATCH_SIZE - 1) // BATCH_SIZE
est_time = num_batches * 3 / 60  # ~3 sec per batch rough estimate
print(f"Estimated: {num_batches} batches, ~{est_time:.0f} min")
print("=" * 60 + "\n")

start_time = time.time()
enhanced_queries = []
errors = []

# Process in batches
for batch_idx in tqdm(range(num_batches), desc="Enhancing queries"):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(query_texts))
    batch_queries = query_texts[start_idx:end_idx]
    batch_qids = query_ids[start_idx:end_idx]

    try:
        batch_enhanced = generate_batch(batch_queries)
        enhanced_queries.extend(batch_enhanced)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            # OOM: fall back to single-query for this batch
            print(f"\nOOM at batch {batch_idx}! Falling back to single-query mode...")
            torch.cuda.empty_cache()
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                try:
                    result = generate_batch([q])
                    enhanced_queries.extend(result)
                except Exception as e2:
                    print(f"  Error on query {start_idx+j} [{qid}]: {e2}")
                    errors.append((start_idx+j, qid, str(e2)))
                    enhanced_queries.append(q)  # Fallback to original
        else:
            # Non-OOM error: log and fall back for entire batch
            print(f"\nError at batch {batch_idx}: {e}")
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                errors.append((start_idx+j, qid, str(e)))
                enhanced_queries.append(q)

    # Checkpoint
    completed = len(enhanced_queries)
    if completed % CHECKPOINT_EVERY < BATCH_SIZE and completed > 0:
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({
                'enhanced_so_far': enhanced_queries,
                'completed': completed,
                'total': len(query_texts)
            }, f)

elapsed = time.time() - start_time
print(f"\nEnhanced {len(enhanced_queries)} queries in {elapsed/60:.1f} minutes")
print(f"  Speed: {len(enhanced_queries) / (elapsed/60):.1f} queries/minute")
print(f"  Batch size: {BATCH_SIZE}")
if errors:
    print(f"  Errors: {len(errors)} (fell back to original query)")
    for idx, qid, err in errors[:5]:
        print(f"    Query {idx} [{qid}]: {err}")

## Step 7: Save Results

In [ ]:
import pickle
from datetime import datetime

quant_mode = '4-bit NF4 (bitsandbytes, double-quant)' if USE_4BIT else 'FP16 (no quantization)'

data = {
    'query_ids': query_ids,
    'original': query_texts,
    'enhanced': enhanced_queries,
    'metadata': {
        'model': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'architecture': 'Standard LlamaForCausalLM (RoPE, SiLU, RMSNorm, Full MHA)',
        'developers': 'NCAI at SDAIA (Saudi Data and AI Authority)',
        'training_tokens': '5.2 trillion (4T English + 1.2T Arabic/English)',
        'paper': 'Bari et al. (2025). ALLaM. ICLR 2025. arXiv:2407.15390',
        'quantization': quant_mode,
        'temperature': TEMPERATURE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'top_p': TOP_P,
        'batch_size': BATCH_SIZE,
        'gpu': torch.cuda.get_device_name(0),
        'technique': 'query2doc',
        'dataset': 'miracl-ar-dev',
        'date': datetime.now().isoformat(),
        'num_queries': len(query_ids),
        'runtime_minutes': round(elapsed / 60, 1),
        'queries_per_minute': round(len(enhanced_queries) / (elapsed / 60), 1),
        'errors': len(errors),
        'system_prompt': SYSTEM_PROMPT,
        'preview_note': 'PREVIEW/ALPHA model -- results may differ from final release',
        'research_doc': 'research_decisions/allam_7b_research.md'
    }
}

# Save locally in Colab
local_path = 'enhanced_queries_allam_7b.pkl'
with open(local_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved locally: {local_path}")

# Save to Google Drive for persistence
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
os.makedirs(drive_base, exist_ok=True)
drive_path = f'{drive_base}/enhanced_queries_allam_7b.pkl'
with open(drive_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved to Drive: {drive_path}")

print(f"\nKey metadata:")
print(f"  Model: {data['metadata']['model']}")
print(f"  Revision: {data['metadata']['model_revision']}")
print(f"  Quantization: {data['metadata']['quantization']}")
print(f"  Batch size: {data['metadata']['batch_size']}")
print(f"  Runtime: {data['metadata']['runtime_minutes']} min")
print(f"  Speed: {data['metadata']['queries_per_minute']} queries/min")

## Step 8: Expansion Statistics

In [ ]:
import numpy as np

print("=" * 60)
print("EXPANSION STATISTICS")
print("=" * 60)

orig_lens = [len(q) for q in query_texts]
enh_lens = [len(q) for q in enhanced_queries]
ratios = [e / max(o, 1) for e, o in zip(enh_lens, orig_lens)]

print(f"\nALLaM-7B-Instruct ({'4-bit' if USE_4BIT else 'FP16'}, temp={TEMPERATURE}):")
print(f"  Avg original length : {np.mean(orig_lens):.1f} chars")
print(f"  Avg enhanced length : {np.mean(enh_lens):.1f} chars")
print(f"  Avg expansion ratio : {np.mean(ratios):.2f}x")
print(f"  Median expansion    : {np.median(ratios):.2f}x")
print(f"  Min expansion       : {np.min(ratios):.2f}x")
print(f"  Max expansion       : {np.max(ratios):.2f}x")

print("\n" + "=" * 60)
print("REFERENCE (previous experiments):")
print("  exp_003 Qwen 2.5 3B:     Avg 9.73x (247.6 chars)")
print("  exp_005 Falcon-H1-3B:    See exp_005 doc")
print("=" * 60)

print(f"\nPkl file saved. Next step:")
print(f"  Open evaluate_enhanced_queries.ipynb")
print(f"  Upload {local_path} for Dense retrieval evaluation")

---

## Results (exp_007) — WORST MODEL, DROP

### Dense Retrieval Results

| Metric | Baseline (mDPR) | ALLaM-7B Dense | vs Baseline |
|--------|-----------------|----------------|-------------|
| NDCG@10 | 0.4993 | **0.2550** | **-48.9%** |
| Recall@10 | 0.6156 | **0.3335** | **-45.8%** |
| Recall@100 | 0.8407 | **0.5465** | **-35.0%** |
| MRR | 0.5328 | **0.2708** | **-49.2%** |

### BM25 Retrieval Results

| Metric | ALLaM-7B BM25 |
|--------|---------------|
| NDCG@10 | 0.3341 |
| Recall@10 | 0.4348 |
| Recall@100 | 0.7004 |
| MRR | 0.3676 |

### All Models Comparison (Dense + Query2Doc)

| Model | NDCG@10 | Recall@10 | MRR | Status |
|-------|---------|-----------|-----|--------|
| Jais-2-8B (exp_006) | **0.6018** | **0.7161** | **0.6356** | BEST |
| Qwen 2.5 3B (exp_003) | 0.5435 | 0.6608 | 0.5742 | Good |
| Falcon-H1-3B (exp_005) | 0.5359 | 0.6484 | 0.5681 | Good |
| Baseline (no QE) | 0.4993 | 0.6156 | 0.5328 | -- |
| **ALLaM-7B (exp_007)** | **0.2550** | **0.3335** | **0.2708** | **DROP** |

### Runtime

| Metric | Value |
|--------|-------|
| GPU | NVIDIA A100-SXM4-40GB |
| Precision | FP16 (no quantization) |
| VRAM used | 14.5 GB (28.0 GB free) |
| Batch size | 16 |
| Runtime | 16.1 min |
| Speed | 179.5 queries/min |
| Errors | 0 |

---

## Lessons Learned

### Technical
1. **FP16 worked fine on A100** — 14.5 GB, 28 GB free, no OOM
2. **Batch size 16 worked** — standard Transformer, no batching issues
3. **token_type_ids removal worked** — same as Jais-2
4. **Runtime was reasonable** — 179.5 queries/min (16.1 min total), faster than Falcon-H1
5. **Preview model ran smoothly** — no crashes, no unexpected errors

### Root Causes of Failure
1. **Sentencepiece `▁` token markers leaking into decoded text** — raw sentencepiece subword markers visible in output (`▁علي ▁بن ▁محمد ▁السم ري`). If literal in decoded strings, this corrupts both BM25 term matching and mDPR BERT tokenization. Likely a tokenizer bug in the preview model.
2. **Severe factual hallucinations** — Query 3 says Paul is "the Rock" (should be Peter). Query 5 says Congo war was 1960 (should be 1996). Wrong terms push retriever toward irrelevant documents.
3. **Wildly inconsistent expansion ratios** — range 3.3x to 23.3x (Jais-2 had median 5.04x). Some queries barely expanded, others flooded with noise.
4. **Preview/alpha model quality** — Ar-IFEval prompt-strict was only 31.34, suggesting weak instruction following. The model may not reliably follow "Respond in Arabic only."

### Research
1. **Training data volume (5.2T tokens, 540B Arabic) did NOT predict QE quality** — ALLaM had more Arabic training data than any other model but produced the worst results
2. **7B (FP16) lost to 3B models** — even Falcon-H1-3B and Qwen 2.5-3B significantly outperformed ALLaM-7B
3. **Preview/alpha status carries real downstream risk** — the model runs fine but generates poor content for retrieval tasks
4. **Tokenizer correctness is critical** — sentencepiece decode bugs destroy the entire retrieval pipeline
5. **Temperature 0.7 was used; 0.6 (model recommended) was not tested** — unlikely to fix the fundamental tokenizer issue

### Key Finding
**ALLaM-7B-Instruct-preview is the only model that degraded retrieval below baseline.** This is a valuable negative result for the thesis: query expansion is not universally beneficial, and model quality (especially tokenizer correctness and instruction following) is critical. The preview/alpha status is a contributing factor. Results may differ with a stable release.

---

## Citations

- Wang, L., Yang, N., & Wei, F. (2023). Query2doc: Query Expansion with Large Language Models. arXiv:2303.07678.
- Bari, M.S., et al. (2025). ALLaM: Large Language Models for Arabic and English. ICLR 2025. arXiv:2407.15390.
- Zhang, X., et al. (2023). MIRACL: A Multilingual Retrieval Dataset. TACL.
- Research notes: `research_decisions/allam_7b_research.md`